In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load files
blank = pd.read_csv('Blank.csv')
sample1 = pd.read_csv('Sample1.csv')

def extract_mean_spectrum(df):
    cols = df.columns.tolist()
    wavelengths = None
    spectra = []
    for i in range(0, len(cols)-1, 2):
        wl = pd.to_numeric(df[cols[i]], errors='coerce')
        ab = pd.to_numeric(df[cols[i+1]], errors='coerce')
        mask = (~wl.isna()) & (~ab.isna())
        wl = wl[mask]
        ab = ab[mask]
        if wavelengths is None:
            wavelengths = wl.values
        spectra.append(ab.values)
    spectra = np.array(spectra)
    mean_spec = np.mean(spectra, axis=0)
    return wavelengths, mean_spec

# Compute mean spectra
wl_blank, mean_blank = extract_mean_spectrum(blank)
wl_sample, mean_sample = extract_mean_spectrum(sample1)

# Interpolate blank to sample wavelengths if needed
if not np.array_equal(wl_blank, wl_sample):
    mean_blank_interp = np.interp(wl_sample, wl_blank, mean_blank)
else:
    mean_blank_interp = mean_blank

# Corrected spectrum
corrected = mean_sample - mean_blank_interp

# Plot raw
plt.figure()
plt.plot(wl_sample, mean_sample)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Absorbance')
plt.title('Sample 1 - Raw (averaged)')
plt.grid()
plt.show()

# Plot corrected
plt.figure()
plt.plot(wl_sample, corrected)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Absorbance (corrected)')
plt.title('Sample 1 - Blank corrected')
plt.grid()
plt.show()